In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Seed for reproducibility
np.random.seed(42)
random.seed(42)

print("⏳ Generating messy healthcare datasets for IPD... Please wait...")

# --- 1. STATE MASTER TABLE (10 Rows) ---
states_data = {
    'StateID': [f'S{i}' for i in range(1, 11)],
    'StateName': ['Andhra Pradesh', 'Telangana', 'Tamil Nadu', 'Karnataka', 'Kerala', 'Maharashtra', 'Delhi', 'Gujarat', 'West Bengal', 'Punjab'],
    'HospitalName': ['Apollo Hospitals', 'Care Hospitals', 'AIMS', 'Fortis Healthcare', 'KIMS', 'Max Hospital', 'Manipal Hospital', 'Narayana Health', 'Medanta', 'Global Hospitals'],
    'HospitalID': [f'H{i}' for i in range(1, 11)],
    'DiabetesCode': [f'DIA{100+i}' for i in range(1, 11)]
}
df_state = pd.DataFrame(states_data)

# --- 2. POPULATION TABLE (15,000 Rows) ---
rows_pop = 15000
user_ids = [f'USR{str(i).zfill(5)}' for i in range(1, rows_pop + 1)]

# Generating messy dates for UserDOB
start_date = datetime(1950, 1, 1)
dob_formats = ['%d-%m-%Y', '%Y-%m-%d', '%m/%d/%Y']
messy_dobs = []
for _ in range(rows_pop):
    random_days = random.randint(0, 22000)
    dt = start_date + timedelta(days=random_days)
    fmt = random.choice(dob_formats)
    messy_dobs.append(dt.strftime(fmt))

diabetes_codes = [f'DIA{random.randint(101, 110)}' for _ in range(rows_pop)]
diabetes_values = np.round(np.random.uniform(70.0, 300.0, size=rows_pop), 1)
parameters = ['Fasting', 'Post-Prandial', 'HbA1c']
diabetes_parameters = [random.choice(parameters) for _ in range(rows_pop)]

# Intentionally making statuses inconsistent/messy
status_options = ['High', 'Low', 'Moderate', 'high ', 'MODERATE', 'low']
diabetes_statuses = [random.choice(status_options) for _ in range(rows_pop)]

df_population = pd.DataFrame({
    'UserID': user_ids,
    'UserDOB': messy_dobs,
    'DiabetesCode': diabetes_codes,
    'DiabetesValue': diabetes_values,
    'DiabetesParameter': diabetes_parameters,
    'DiabetesStatus': diabetes_statuses
})

# --- 3. HOSPITAL TABLE (35 Rows) ---
rows_hosp = 35
hosp_user_ids = [f'USR{str(random.randint(1, rows_pop)).zfill(5)}' for _ in range(rows_hosp)]
hosp_ids = [f'H{random.randint(1, 10)}' for _ in range(rows_hosp)]

# Generating messy appointment dates
appt_dates = []
for _ in range(rows_hosp):
    dt = datetime(2026, 1, 1) + timedelta(days=random.randint(0, 120))
    appt_dates.append(dt.strftime(random.choice(['%d-%m-%Y %H:%M', '%Y/%m/%d'])))

appt_times = [f'{str(random.randint(9,17)).zfill(2)}:{str(random.choice([0,15,30,45])).zfill(2)}' for _ in range(rows_hosp)]
doctor_ids = [f'DOC{random.randint(501, 515)}' for _ in range(rows_hosp)]
hosp_statuses = [random.choice(['High', 'Low', 'Moderate']) for _ in range(rows_hosp)]

df_hospital = pd.DataFrame({
    'HospitalID': hosp_ids,
    'UserID': hosp_user_ids,
    'ApptDate': appt_dates,
    'ApptTime': appt_times,
    'DoctorID': doctor_ids,
    'DiabetesStatus': hosp_statuses
})

print("✅ All 3 Tables Generated Successfully!")
print(f"-> Population Table Shape: {df_population.shape}")
print(f"-> State Table Shape: {df_state.shape}")
print(f"-> Hospital Table Shape: {df_hospital.shape}\n")
print("Data variables ready: 'df_population', 'df_state', 'df_hospital'")

⏳ Generating messy healthcare datasets for IPD... Please wait...
✅ All 3 Tables Generated Successfully!
-> Population Table Shape: (15000, 6)
-> State Table Shape: (10, 5)
-> Hospital Table Shape: (35, 6)

Data variables ready: 'df_population', 'df_state', 'df_hospital'


In [4]:
# Just checking the messy nature of the status and date columns
df_population[['UserID',	'UserDOB',	'DiabetesCode',	'DiabetesValue',	'DiabetesParameter',	'DiabetesStatus']].head(10)

,UserID,UserDOB,DiabetesCode,DiabetesValue,DiabetesParameter,DiabetesStatus
0,USR00001,14-05-2007,DIA105,156.1,Fasting,MODERATE
1,USR00002,03/30/1952,DIA101,288.7,HbA1c,low
2,USR00003,04-09-1974,DIA108,238.4,Post-Prandial,low
3,USR00004,10-01-1970,DIA109,207.7,HbA1c,MODERATE
4,USR00005,03/13/1959,DIA109,105.9,Fasting,Low
5,USR00006,05-12-1998,DIA101,105.9,Post-Prandial,MODERATE
6,USR00007,2002-12-23,DIA103,83.4,Post-Prandial,low
7,USR00008,07-11-1952,DIA107,269.2,Post-Prandial,Low
8,USR00009,29-05-1958,DIA106,208.3,HbA1c,low
9,USR00010,11/15/1970,DIA103,232.9,HbA1c,low


In [5]:
df_population['DiabetesStatus'] = df_population['DiabetesStatus'].str.strip().str.capitalize()

df_hospital['DiabetesStatus'] = df_hospital['DiabetesStatus'].str.strip().str.capitalize()

print(df_population['DiabetesStatus'].unique())

print(df_hospital['DiabetesStatus'].unique())

['Moderate' 'Low' 'High']
['Low' 'High' 'Moderate']


In [6]:
# 1. Processing mixed date strings into clean datetime data objects
df_population['UserDOB'] = pd.to_datetime(df_population['UserDOB'], format='mixed')

# 2. Let's verify the updated standard properties output datatype check
print("Updated DataType Verification:")
print(df_population['UserDOB'].dtypes)

# 3. Printing sample output configuration check uniformity pattern
df_population[['UserID', 'UserDOB']].head(10)

Updated DataType Verification:
datetime64[ns]


,UserID,UserDOB
0,USR00001,2007-05-14
1,USR00002,1952-03-30
2,USR00003,1974-04-09
3,USR00004,1970-10-01
4,USR00005,1959-03-13
5,USR00006,1998-05-12
6,USR00007,2002-12-23
7,USR00008,1952-07-11
8,USR00009,1958-05-29
9,USR00010,1970-11-15


In [7]:
df_hospital['ApptDate'] = pd.to_datetime(df_hospital['ApptDate'], format='mixed')

print("Updated DataType Verification:")
print(df_hospital['ApptDate'].dtypes)

df_hospital[['HospitalID', 'ApptDate']].head()

Updated DataType Verification:
datetime64[ns]


,HospitalID,ApptDate
0,H2,2026-10-03
1,H8,2026-02-24
2,H3,2026-07-03
3,H7,2026-03-21
4,H2,2026-12-02


In [8]:
# 1. Checking Missing Values (NaN) in all 3 tables
print("=== Missing Values Audit ===")
print("\n--- Population Table ---")
print(df_population.isnull().sum())

print("\n--- Hospital Table ---")
print(df_hospital.isnull().sum())

print("\n--- State Table ---")
print(df_state.isnull().sum())

print("\n" + "="*30 + "\n")

# 2. Checking Duplicate Rows based on Primary Keys
print("=== Duplicates Audit ===")
print(f"Duplicate UserIDs in Population: {df_population['UserID'].duplicated().sum()}")
print(f"Duplicate HospitalIDs in State: {df_state['HospitalID'].duplicated().sum()}")

=== Missing Values Audit ===

--- Population Table ---
UserID               0
UserDOB              0
DiabetesCode         0
DiabetesValue        0
DiabetesParameter    0
DiabetesStatus       0
dtype: int64

--- Hospital Table ---
HospitalID        0
UserID            0
ApptDate          0
ApptTime          0
DoctorID          0
DiabetesStatus    0
dtype: int64

--- State Table ---
StateID         0
StateName       0
HospitalName    0
HospitalID      0
DiabetesCode    0
dtype: int64


=== Duplicates Audit ===
Duplicate UserIDs in Population: 0
Duplicate HospitalIDs in State: 0


In [9]:
# Exporting cleaned active RAM engine datasets into professional database CSV files
df_population.to_csv('Clean_Population_Master.csv', index=False)
df_state.to_csv('Clean_State_Master.csv', index=False)
df_hospital.to_csv('Clean_Hospital_Master.csv', index=False)

print("🎯 Clean Data Files Successfully Generated for MySQL Database Engine Ingestion!")
print("Files ready in your Colab files panel: 'Clean_Population_Master.csv', 'Clean_State_Master.csv', 'Clean_Hospital_Master.csv'")

from google.colab import files
files.download('Clean_Population_Master.csv')
files.download('Clean_State_Master.csv')
files.download('Clean_Hospital_Master.csv')

🎯 Clean Data Files Successfully Generated for MySQL Database Engine Ingestion!
Files ready in your Colab files panel: 'Clean_Population_Master.csv', 'Clean_State_Master.csv', 'Clean_Hospital_Master.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import pandas as pd
import random
from datetime import datetime, timedelta

# 1. Base Parameter Arrays Setup
user_ids = list(range(36, 66))  # 30 new users (36 to 65)
states = ['Telangana', 'Andhra Pradesh', 'Maharashtra', 'Delhi', 'Gujarat']
hospitals = [
    ('H101', 'Apollo Hospitals'),
    ('H102', 'Fortis Healthcare'),
    ('H103', 'KIMS'),
    ('H104', 'AIMS'),
    ('H105', 'Medanta')
]
diabetes_types = [
    ('FPG', 'Fasting Plasma Glucose', 'mg/dL'),
    ('PPBG', 'Postprandial Blood Glucose', 'mg/dL'),
    ('HbA1c', 'Hemoglobin A1c', '%')
]

# --- POPULATION TABLE GENERATION ---
population_data = []
for uid in user_ids:
    dob = (datetime(1955, 1, 1) + timedelta(days=random.randint(0, 18000))).strftime('%Y-%m-%d')
    dtype = random.choice(diabetes_types)
    d_value = round(random.uniform(5.0, 15.0), 2) if dtype[0] == 'HbA1c' else random.randint(80, 250)

    # Status logic matching values
    if dtype[0] == 'HbA1c':
        status = 'Normal' if d_value < 5.7 else 'Pre-Diabetes' if d_value <= 6.4 else 'Critical'
    else:
        status = 'Normal' if d_value < 100 else 'Pre-Diabetes' if d_value <= 125 else 'Critical'

    population_data.append([uid, dob, dtype[0], d_value, dtype[1], status])

df_population = pd.DataFrame(population_data, columns=['UserID', 'UserDOB', 'DiabetesCode', 'DiabetesValue', 'DiabetesParameter', 'DiabetesStatus'])

# --- STATE TABLE GENERATION ---
state_data = []
for i, (hid, hname) in enumerate(hospitals):
    state_data.append([i+1, random.choice(states), hname, hid, random.choice(['FPG', 'PPBG', 'HbA1c'])])

df_state = pd.DataFrame(state_data, columns=['StateID', 'StateName', 'HospitalName', 'HospitalID', 'DiabetesCode'])

# --- HOSPITAL TABLE GENERATION ---
hospital_data = []
for uid in user_ids:
    hid, _ = random.choice(hospitals)
    appt_date = (datetime(2026, 5, 1) + timedelta(days=random.randint(0, 25))).strftime('%Y-%m-%d')
    appt_time = f"{random.randint(9, 17)}:{random.choice(['00', '15', '30', '45'])}"
    doc_id = f"DOC{random.randint(201, 210)}"
    p_status = df_population[df_population['UserID'] == uid]['DiabetesStatus'].values[0]

    hospital_data.append([hid, uid, appt_date, appt_time, doc_id, p_status])

df_hospital = pd.DataFrame(hospital_data, columns=['HospitalID', 'UserID', 'ApptDate', 'ApptTime', 'DoctorID', 'DiabetesStatus'])

# Save to CSV files for downloading
df_population.to_csv('New_Population_Table.csv', index=False)
df_state.to_csv('New_State_Table.csv', index=False)
df_hospital.to_csv('New_Hospital_Table.csv', index=False)

print("Superb! 3 Files successfully generated according to image_c334dd structures.")

Superb! 3 Files successfully generated according to image_c334dd structures.


In [11]:
from google.colab import files
files.download('New_Population_Table.csv')
files.download('New_State_Table.csv')
files.download('New_Hospital_Table.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>